# Bulk ATAC-seq part 2: consensus peaks and MACS3 differential peaks

**Case study: How do FOXA1 and HNF4A change the chromatin of a blood cell line?**

Hansen JL, Loell KJ, Cohen BA. *A test of the pioneer factor hypothesis using ectopic liver gene activation.* eLife (2022) 11:e73358. https://doi.org/10.7554/eLife.73358

**Research questions:**

1. How good are the ATAC-seq libraries?
2. Which regions open (or close) when FOXA1 and HNF4A are expressed?
3. Do the opened regions contain FOXA1 and HNF4A motifs, and do they lie near the genes that are switched on?

In this notebook we prepare the analysis of question 2 (consensus peaks and counts) and see a first, simple answer with MACS3; [part 3](part_3_diffbind_deseq2.ipynb) answers questions 2 and 3 with statistics that use the replicates.

## Introduction

In [part 1](part_1_reads_to_peaks.ipynb) we called peaks for two samples and saw that one sample versus one sample cannot tell us whether accessibility changes. Now we use all **four samples** (uninduced and induced, 2 replicates each) of [GSE182188](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE182188).

To compare samples, we first need **one common set of regions** in which we count fragments for every sample. Peaks called per sample are not directly comparable: the same open region gets slightly different boundaries in each sample, and weak peaks are called in some samples but not in others. A **consensus peak set** solves this. There are several ways to build one, and the choice changes the regions you test, so we compare three:

1. the **union** of all peaks, merged where they overlap;
2. peaks **supported by at least *N* samples**;
3. **fixed-width peaks around summits**, selected iteratively by significance (the method of Corces et al. 2018, also used by ArchR and SnapATAC2).

We then **count fragments** per sample in the consensus peaks: this count matrix is the input for DiffBind and DESeq2 in [part 3](part_3_diffbind_deseq2.ipynb).

Finally, we try a peak caller's own differential mode, **MACS3 `bdgdiff`**, and see why it is not enough on its own.

All files are restricted to chr17 so everything runs in minutes; the steps are identical genome-wide.

## Step 0: Setup

This notebook uses the `atac-cli` environment and the "Bash (atac-cli)" kernel; see [part 1](part_1_reads_to_peaks.ipynb) for how to install them. The course data are in the shared folder on the teaching server, or in the folder `course_data` next to this notebook (see Step 0 of part 1); we work in `data/part_2`, next to this notebook.

In [4]:
# The folder of this notebook (the kernel starts there). Kept when you run this cell again after the cd below.
HERE=${HERE:-$PWD}

# Course data (read-only): the shared folder on the teaching server, otherwise the folder course_data next to this notebook
BULK=/data/teachers/data/bulk_atac_seq/hansen_foxa1_hnf4a
[ -d $BULK ] || BULK=$HERE/course_data
GENOME=$BULK/genome/hg38

# Your own working folder, in the folder data next to this notebook
OUT=$HERE/data/part_2
mkdir -p $OUT && cd $OUT

THREADS=4
# the four samples of the sample sheet of part 1: uninduced (U) and induced (I), two replicates each
SAMPLES="FOXA1HNF4A_U1 FOXA1HNF4A_U2 FOXA1HNF4A_I1 FOXA1HNF4A_I2"
echo "samples: $SAMPLES"

# effective size of chr17 (non-N bases), for MACS3
GSIZE=$(samtools faidx $GENOME/chr17.fa chr17 | grep -v '>' | tr -d 'Nn\n' | wc -c)
echo "chr17 effective size: $GSIZE"

samples: FOXA1HNF4A_U1 FOXA1HNF4A_U2 FOXA1HNF4A_I1 FOXA1HNF4A_I2
chr17 effective size: 82920204


## Step 1: The filtered BAM files

The four samples were processed exactly like in [part 1](part_1_reads_to_peaks.ipynb) (trimming, alignment, duplicate marking, filtering) on the full data, with `scripts/10_bulk_pipeline.sh`. We link the chr17 part of each filtered BAM:

In [15]:
mkdir -p bam
# ln -sf: a symbolic link instead of a copy (saves disk space); we also need the index (.bai)
for s in $SAMPLES; do
    ln -sf $BULK/data_generated/$s/$s.filtered.chr17.bam bam/$s.bam
    ln -sf $BULK/data_generated/$s/$s.filtered.chr17.bam.bai bam/$s.bam.bai
done
# -f 64: count only the first read of each pair, i.e. fragments
for s in $SAMPLES; do
    printf "%-14s %9d fragments on chr17\n" $s $(samtools view -c -f 64 bam/$s.bam)
done

FOXA1HNF4A_U1    1550647 fragments on chr17
FOXA1HNF4A_U2    1617874 fragments on chr17
FOXA1HNF4A_I1    1814127 fragments on chr17
FOXA1HNF4A_I2    1633830 fragments on chr17


## Step 2: Peaks per sample

We call peaks for each sample with the same MACS3 settings as in part 1 (paired-end mode, q < 0.01). This takes a few minutes.

In [16]:
mkdir -p peaks
for s in $SAMPLES; do
    macs3 callpeak -f BAMPE -t bam/$s.bam -n $s -g $GSIZE --keep-dup all -q 0.01 \
        --outdir peaks 2> peaks/$s.log
done
wc -l peaks/*_peaks.narrowPeak

   5198 peaks/FOXA1HNF4A_I1_peaks.narrowPeak
   4923 peaks/FOXA1HNF4A_I2_peaks.narrowPeak
   3520 peaks/FOXA1HNF4A_U1_peaks.narrowPeak
   3706 peaks/FOXA1HNF4A_U2_peaks.narrowPeak
  17347 total


*If peak calling takes too long, use the pre-generated peaks: `cp -r $BULK/nb02_generated/peaks .`*

You should get about 3,500-3,700 peaks per uninduced sample and 4,900-5,200 per induced sample on chr17. The replicates agree well, and the difference between the conditions is much larger than between the replicates.

How similar are the peak sets? The **Jaccard index** of two sets of regions is the number of base pairs they share divided by the number of base pairs covered by either (0 = no overlap, 1 = identical). Let's compare the two replicate pairs with a pair from different conditions:

In [17]:
jaccard() {  # jaccard index of the peaks of two samples
    # <( ... ): pass the output of a command as if it were a file; bedtools jaccard needs sorted input
    # the last line of the output holds the numbers; column 3 is the Jaccard index
    bedtools jaccard -a <(sort -k1,1 -k2,2n peaks/$1_peaks.narrowPeak) \
                     -b <(sort -k1,1 -k2,2n peaks/$2_peaks.narrowPeak) | tail -1 | cut -f3
}
echo "U1 vs U2 (replicates, uninduced): $(jaccard FOXA1HNF4A_U1 FOXA1HNF4A_U2)"
echo "I1 vs I2 (replicates, induced):   $(jaccard FOXA1HNF4A_I1 FOXA1HNF4A_I2)"
echo "U1 vs I1 (different condition):   $(jaccard FOXA1HNF4A_U1 FOXA1HNF4A_I1)"

U1 vs U2 (replicates, uninduced): 0.726998
I1 vs I2 (replicates, induced):   0.734553
U1 vs I1 (different condition):   0.450294


The replicates share about 73% of their peak base pairs, an uninduced and an induced sample only 45%. 

**Question**: Why is the Jaccard index between conditions so much lower, if the induced cells mostly *gain* peaks? (Hint: the index divides by the base pairs covered by *either* sample.) Exercise 1 at the end of this notebook compares all pairs of samples.

## Step 3: Consensus peaks

### 3a. Union of all peaks

The simplest consensus: pool the peaks of all samples and merge everything that overlaps (`bedtools merge`).

In [19]:
# pool the peaks of all samples, keep chr/start/end, sort by position (bedtools merge needs sorted input) and merge overlaps
cat peaks/*_peaks.narrowPeak | cut -f1-3 | sort -k1,1 -k2,2n | bedtools merge > consensus_union.bed

widths() {  # summary of region widths in a BED file
    awk '{print $3 - $2}' $1 | sort -n | awk '{w[NR] = $1; s += $1}
        END {printf "%6d regions, width min %d / median %d / max %d bp, %.1f Mb in total\n", NR, w[1], w[int(NR/2)+1], w[NR], s/1e6}'
}
echo "per-sample peaks (all 4 pooled): $(cat peaks/*_peaks.narrowPeak | wc -l)"
echo -n "union: "; widths consensus_union.bed

per-sample peaks (all 4 pooled): 17347
union:   6469 regions, width min 203 / median 536 / max 4233 bp, 4.2 Mb in total


Look at the maximum width. Merging "daisy-chains" peaks: if sample A has a peak that overlaps one of sample B, which overlaps one of sample C, all three become one long region. A broad region can hide a change in one of its parts. The union also keeps every peak that was called in only one sample, which is often noise.

### 3b. Peaks supported by at least *N* samples

We can require that a merged region overlaps peaks from at least *N* different samples. This is what DiffBind does by default (`minOverlap = 2`). `bedtools merge -c 4 -o count_distinct` counts the distinct sample names in each merged region:

In [20]:
# add the sample name as column 4 to every peak, pool and merge; -c 4 -o count_distinct: number of different samples per region
for s in $SAMPLES; do
    awk -v s=$s 'BEGIN {OFS = "\t"} {print $1, $2, $3, s}' peaks/${s}_peaks.narrowPeak
done | sort -k1,1 -k2,2n | bedtools merge -c 4 -o count_distinct > union_with_support.bed

echo "N  regions supported by >= N samples"
for n in 1 2 3 4; do
    printf "%-2d %d\n" $n $(awk -v n=$n '$4 >= n' union_with_support.bed | wc -l)
done
awk '$4 >= 2' union_with_support.bed | cut -f1-3 > consensus_min2.bed
echo -n "min 2 samples: "; widths consensus_min2.bed

N  regions supported by >= N samples
1  6469
2  5224
3  2956
4  2428
min 2 samples:   5224 regions, width min 211 / median 621 / max 4233 bp, 3.8 Mb in total


**Question**: Why should *N* not be larger than 2 in this experiment? (Hint: think of a region that opens only in the induced cells.)

### 3c. Fixed-width peaks around summits, selected iteratively

The third strategy avoids both long merged regions and arbitrary boundaries:

1. take the **summit** of every peak in every sample (the position with the highest pileup, `*_summits.bed`) and extend it to a fixed **500 bp** window;
2. give each window a score that is comparable between samples: its -log10 q-value divided by the sum over all summits of that sample, times 10<sup>6</sup> ("score per million");
3. keep the highest-scoring window, remove all windows that overlap it, and repeat until no windows are left.

The result is a set of non-overlapping 500 bp regions, each centred on the strongest summit in its neighbourhood. In bash, step 3 can be done in rounds: `bedtools cluster` groups overlapping windows, we keep the best of each group, and remove everything that overlaps a kept window.

In [21]:
# 1-2. 500 bp windows around all summits, with a normalised score (column 5 of *_summits.bed is -log10 q)
for s in $SAMPLES; do
    awk -v s=$s 'BEGIN {OFS = "\t"} {line[NR] = $0; total += $5}
        END {for (i = 1; i <= NR; i++) {
                 split(line[i], f, "\t"); start = f[2] - 250; if (start < 0) start = 0
                 print f[1], start, f[2] + 250, s "_" i, f[5] / total * 1e6 }}' peaks/${s}_summits.bed
done | sort -k1,1 -k2,2n > summit_windows.bed
echo "$(wc -l < summit_windows.bed) summit windows"

# 3. iterative selection
cp summit_windows.bed remaining.bed; : > selected.bed; round=0
while [ -s remaining.bed ]; do
    round=$(( round + 1 ))
    bedtools cluster -i remaining.bed | sort -k6,6n -k5,5gr | awk '!seen[$6]++' | cut -f1-5 \
        | sort -k1,1 -k2,2n > best.bed
    cat best.bed >> selected.bed
    bedtools intersect -v -a remaining.bed -b best.bed > tmp.bed && mv tmp.bed remaining.bed
    echo "round $round: kept $(wc -l < best.bed), $(wc -l < remaining.bed) windows left"
done
sort -k1,1 -k2,2n selected.bed > consensus_fixed500_all.bed
rm remaining.bed best.bed selected.bed
echo -n "fixed width: "; widths consensus_fixed500_all.bed

17347 summit windows
round 1: kept 6720, 43 windows left
round 2: kept 36, 0 windows left
fixed width:   6756 regions, width min 500 / median 500 / max 500 bp, 3.4 Mb in total


Like the union, this set still contains windows whose summit was found in a single sample. We combine it with the idea of 3b: keep only windows that overlap peaks of **at least 2 samples**.

In [22]:
cut -f1-4 consensus_fixed500_all.bed > windows.tmp
# for each sample a 0/1 column: does the window overlap a peak of that sample? (intersect -c counts the overlaps)
for s in $SAMPLES; do
    bedtools intersect -c -a windows.tmp -b peaks/${s}_peaks.narrowPeak | awk '{print ($5 > 0)}' > support_$s.tmp
done
paste windows.tmp support_*.tmp | awk 'BEGIN {OFS = "\t"} {n = 0; for (i = 5; i <= NF; i++) n += $i; if (n >= 2) print $1, $2, $3, "peak" NR}' \
    > consensus_peaks.bed
rm windows.tmp support_*.tmp
echo -n "fixed width, >= 2 samples: "; widths consensus_peaks.bed

fixed width, >= 2 samples:   5516 regions, width min 500 / median 500 / max 500 bp, 2.8 Mb in total


### Comparing the three consensus sets

Load `consensus_union.bed`, `consensus_min2.bed` and `consensus_peaks.bed` in IGV together with a few per-sample peak files (go to *CCL16* or *ASGR1* again). How do the regions differ?

Here is the overview:

In [23]:
for f in consensus_union.bed consensus_min2.bed consensus_fixed500_all.bed consensus_peaks.bed; do
    printf "%-28s" $f; widths $f
done

consensus_union.bed           6469 regions, width min 203 / median 536 / max 4233 bp, 4.2 Mb in total
consensus_min2.bed            5224 regions, width min 211 / median 621 / max 4233 bp, 3.8 Mb in total
consensus_fixed500_all.bed    6756 regions, width min 500 / median 500 / max 500 bp, 3.4 Mb in total
consensus_peaks.bed           5516 regions, width min 500 / median 500 / max 500 bp, 2.8 Mb in total


There is no single correct choice, but for differential analysis the fixed-width set has clear advantages: every region has the same size (so counts are comparable between regions), regions are centred on the summit (where the change usually is), and there is no daisy-chaining. It is also the standard input for downstream tools such as motif enrichment or sequence-based models. We continue with `consensus_peaks.bed`.

## Step 4: Counting fragments in the consensus peaks

[featureCounts](https://subread.sourceforge.net/featureCounts.html) (from the Subread package) counts reads or fragments per region. It reads regions in **SAF** format (GeneID, Chr, Start, End, Strand; 1-based coordinates):

In [24]:
# BED is 0-based (start not included), SAF is 1-based: add 1 to the start
awk 'BEGIN {OFS = "\t"; print "GeneID", "Chr", "Start", "End", "Strand"} {print $4, $1, $2 + 1, $3, "."}' \
    consensus_peaks.bed > consensus_peaks.saf
head -3 consensus_peaks.saf

GeneID	Chr	Start	End	Strand
peak1	chr17	143718	144217	.
peak3	chr17	233287	233786	.


Options:

- `-p --countReadPairs`: count **fragments** (read pairs), not single reads;
- `-O`: a fragment overlapping two neighbouring windows is counted in both (otherwise it is discarded as "ambiguous");
- `-F SAF -a`: the regions.

In [25]:
# -T: threads; -o: output table (+ a .summary file); the BAM files of all samples follow
featureCounts -T $THREADS -p --countReadPairs -O -F SAF -a consensus_peaks.saf \
    -o featurecounts.txt $(for s in $SAMPLES; do echo bam/$s.bam; done) 2> featurecounts.log
tail -4 featurecounts.log
# a clean matrix: region id + one column per sample (featureCounts writes 6 annotation columns before the counts)
( printf "peak"; for s in $SAMPLES; do printf "\t%s" $s; done; echo
  tail -n +3 featurecounts.txt | cut -f1,7- ) > counts_matrix.tsv
head -5 counts_matrix.tsv | column -t

|| ry"                                                                        ||
||                                                                            ||
\\============================================================================//

peak   FOXA1HNF4A_U1  FOXA1HNF4A_U2  FOXA1HNF4A_I1  FOXA1HNF4A_I2
peak1  52             45             77             49
peak3  18             24             105            107
peak6  30             38             70             62
peak7  11             18             71             39


The `.summary` file tells how many fragments were assigned to a region. Assigned / total is the **FRiP** of each sample in the consensus set. Let's compute it for every sample: are they similar?

In [26]:
# the .summary table has one row per category (Assigned, Unassigned_NoFeatures, ...) and one column per sample:
# FRiP = Assigned / sum of all categories
awk -v samples="$SAMPLES" 'BEGIN {n = split(samples, name, " ")}
    NR > 1 {for (i = 2; i <= NF; i++) {total[i] += $i; if ($1 == "Assigned") assigned[i] = $i}}
    END {for (i = 2; i <= n + 1; i++) printf "%-14s FRiP %.3f\n", name[i-1], assigned[i] / total[i]}' featurecounts.txt.summary

FOXA1HNF4A_U1  FRiP 0.416
FOXA1HNF4A_U2  FRiP 0.467
FOXA1HNF4A_I1  FRiP 0.523
FOXA1HNF4A_I2  FRiP 0.525


FRiP in the consensus peaks is about 0.42-0.47 in the uninduced samples and 0.52 in the induced ones: even on one common set of regions, the induced cells have more of their fragments in accessible regions. Here this is (mostly) biology, not library quality: the induced cells have more accessible regions, so more fragments fall in them.

This matters for normalisation. Normalising by the total number of fragments (library size) keeps this difference: the induced samples look "more accessible" in many regions. Methods that assume that most regions do not change (such as DESeq2's median-of-ratios, or normalising by the fragments *in peaks*) partly remove it. Which is right depends on what you believe about the biology. We compare both in [part 3](part_3_diffbind_deseq2.ipynb).

## Step 5: MACS3 differential peaks (`bdgdiff`)

MACS3 can compare two conditions directly. `macs3 bdgdiff` takes, for each condition, the **pileup** of fragments and the **local background (lambda)** as bedGraph files (written by `callpeak -B`), and reports regions where one condition is significantly higher than the other, using a log-likelihood ratio (LLR).

It works on one track per condition, so we **pool** the two replicates of each condition into a single `callpeak` run, and compare induced with uninduced.

In [5]:
mkdir -p pooled
for g in U I; do
    # two BAM files after -t are pooled into one sample; -B also writes the pileup and lambda bedGraphs for bdgdiff
    macs3 callpeak -f BAMPE -t bam/FOXA1HNF4A_${g}1.bam bam/FOXA1HNF4A_${g}2.bam -n $g -g $GSIZE \
        --keep-dup all -q 0.01 -B --outdir pooled 2> pooled/$g.log
done
# -g -G: don't show the owner and group of the files
ls -lhgG pooled/*.bdg

-rw-r--r-- 1 121M Oct 29 16:06 pooled/I_control_lambda.bdg
-rw-r--r-- 1 117M Oct 29 16:06 pooled/I_treat_pileup.bdg
-rw-r--r-- 1 118M Oct 29 16:05 pooled/U_control_lambda.bdg
-rw-r--r-- 1 119M Oct 29 16:05 pooled/U_treat_pileup.bdg


`bdgdiff` needs the sequencing depth of each condition (`--d1`, `--d2`) to scale the two tracks to each other. MACS3 writes it in the header of `*_peaks.xls`; instead of copying the numbers by hand, we read them from the file:

In [28]:
depth() {  # number of fragments used by callpeak, from the .xls header
    # "after filtering" is only written when MACS3 removed duplicates; with --keep-dup all the total is used
    awk -F': ' '/^# (fragments|tags) after filtering in treatment/ {f = $2}
                /^# total (fragments|tags) in treatment/ {t = $2}
                END {print (f ? f : t)}' $1
}
D_I=$(depth pooled/I_peaks.xls)
D_U=$(depth pooled/U_peaks.xls)
echo "induced: $D_I fragments, uninduced: $D_U fragments"

induced: 3447957 fragments, uninduced: 3168521 fragments


Other options: `-l` is the minimum length of a differential region, `-g` the maximum gap to merge nearby regions, `-C` the LLR cutoff (default 3, i.e. the difference is 10<sup>3</sup> = 1000 times more likely than no difference). bdgdiff writes three files: regions higher in condition 1 (`cond1`), higher in condition 2 (`cond2`), and similar in both (`common`).

In [29]:
mkdir -p bdgdiff
# condition 1 = induced, condition 2 = uninduced: --t = treatment pileup, --c = control lambda (local background), --d = depth
macs3 bdgdiff \
    --t1 pooled/I_treat_pileup.bdg --c1 pooled/I_control_lambda.bdg \
    --t2 pooled/U_treat_pileup.bdg --c2 pooled/U_control_lambda.bdg \
    --d1 $D_I --d2 $D_U -l 120 -g 60 -C 3 \
    --outdir bdgdiff --o-prefix induced_vs_uninduced 2> bdgdiff/induced_vs_uninduced.log
wc -l bdgdiff/induced_vs_uninduced_*.bed

  3084 bdgdiff/induced_vs_uninduced_c3.0_common.bed
  3402 bdgdiff/induced_vs_uninduced_c3.0_cond1.bed
  2122 bdgdiff/induced_vs_uninduced_c3.0_cond2.bed
  8608 total


Each file has a header line, so the number of regions is one less than the line count. The strongest differences (column 5 is the LLR score):

In [30]:
# tail -n +2: skip the header line; sort on column 5 (the LLR score), highest first
echo "== more accessible in induced";   tail -n +2 bdgdiff/induced_vs_uninduced_c3.0_cond1.bed | sort -k5,5gr | head -5
echo "== more accessible in uninduced"; tail -n +2 bdgdiff/induced_vs_uninduced_c3.0_cond2.bed | sort -k5,5gr | head -5

== more accessible in induced
chr17	35972936	35974327	induced_vs_uninduced_cond1_1241	650.539
chr17	37147159	37147991	induced_vs_uninduced_cond1_1284	612.125
chr17	31803533	31804159	induced_vs_uninduced_cond1_1075	571.824
chr17	57937857	57938571	induced_vs_uninduced_cond1_2278	482.487
chr17	76173875	76174666	induced_vs_uninduced_cond1_3046	461.246
== more accessible in uninduced
chr17	58328239	58330508	induced_vs_uninduced_cond2_1406	212.641
chr17	42312228	42313304	induced_vs_uninduced_cond2_931	184.68
chr17	42319744	42320476	induced_vs_uninduced_cond2_933	174.383
chr17	49739476	49740580	induced_vs_uninduced_cond2_1239	163.124
chr17	33874556	33875043	induced_vs_uninduced_cond2_639	162.07


The strongest induced region (chr17:35,972,936-35,974,327) lies a few kb from the *CCL16* promoter we saw in part 1.

### How reliable is this?

Pooling throws away the information that makes statistics possible: **how much do replicates of the same condition differ?** A simple test of any differential method is to compare two samples **of the same condition**: any region it reports is a false positive.

Let's run `callpeak -B` on the two uninduced replicates, `FOXA1HNF4A_U1` and `FOXA1HNF4A_U2`, separately and compare them with `bdgdiff`, using the same settings, and put the numbers next to those of induced vs uninduced:

In [31]:
mkdir -p single
# call peaks on each replicate separately, again with -B to write the pileup and lambda tracks
for s in FOXA1HNF4A_U1 FOXA1HNF4A_U2; do
    macs3 callpeak -f BAMPE -t bam/$s.bam -n $s -g $GSIZE --keep-dup all -q 0.01 -B --outdir single 2> single/$s.log
done
# the same bdgdiff settings as above; the depths again come from the .xls headers
macs3 bdgdiff \
    --t1 single/FOXA1HNF4A_U1_treat_pileup.bdg --c1 single/FOXA1HNF4A_U1_control_lambda.bdg \
    --t2 single/FOXA1HNF4A_U2_treat_pileup.bdg --c2 single/FOXA1HNF4A_U2_control_lambda.bdg \
    --d1 $(depth single/FOXA1HNF4A_U1_peaks.xls) --d2 $(depth single/FOXA1HNF4A_U2_peaks.xls) -l 120 -g 60 -C 3 \
    --outdir bdgdiff --o-prefix U1_vs_U2 2> bdgdiff/U1_vs_U2.log
# number of regions per file (minus the header line)
for f in bdgdiff/U1_vs_U2_c3.0_cond?.bed bdgdiff/induced_vs_uninduced_c3.0_cond?.bed; do
    printf "%-45s %6d regions\n" $f $(( $(wc -l < $f) - 1 ))
done

bdgdiff/U1_vs_U2_c3.0_cond1.bed                  110 regions
bdgdiff/U1_vs_U2_c3.0_cond2.bed                  361 regions
bdgdiff/induced_vs_uninduced_c3.0_cond1.bed     3401 regions
bdgdiff/induced_vs_uninduced_c3.0_cond2.bed     2121 regions


Two replicates of the same condition already give about 470 "differential" regions, compared with about 5,500 for induced vs uninduced. Notice also the direction: three quarters of them are "higher" in U2, the uninduced replicate with the higher FRiP (Step 4). Differences between libraries look like differential accessibility. The induced vs uninduced difference is much larger than this noise, so most of its ~3,400 "induced" regions are probably real. But `bdgdiff` gives us no way to tell which ones: it has no estimate of the variation between replicates and no false discovery rate across regions.

There is a second warning sign: about 2,100 regions are "more accessible in uninduced". `bdgdiff` scales the two conditions by their total number of fragments. Just as for *ACTG1* in part 1, if the induced cells open many new regions, every unchanged region gets a smaller share of the fragments and can look "closed". Some of these 2,100 regions may really close, others only look closed after scaling. Part 3 compares normalisations to find out.

`bdgdiff` is still useful to explore a comparison quickly, or when you really have no replicates; the [ChIP-seq practical](../chip-seq/index.md) uses it that way, to compare H3K27ac between one liver and one spleen sample. With replicates, use a method that models the variation between replicates: that is what **DiffBind** and **DESeq2** do in [part 3](part_3_diffbind_deseq2.ipynb), starting from the count matrix we just built.

## Summary

| file | content | used in |
|---|---|---|
| `consensus_peaks.bed` / `.saf` | 500 bp consensus peaks, supported by ≥ 2 samples | part 3 |
| `counts_matrix.tsv` | fragments per consensus peak (rows) and sample (columns) | part 3 (DESeq2) |
| `peaks/*_peaks.narrowPeak` | per-sample peaks | part 3 (DiffBind) |
| `bdgdiff/induced_vs_uninduced_*.bed` | MACS3 differential regions | part 3 (comparison) |

## Exercises

1. Compute the Jaccard index for all pairs of samples and print it as a 4 × 4 table (use the `jaccard` function of Step 2 in two nested loops). Are replicates more similar to each other than to the other condition? How does the Jaccard index relate to the number of peaks per sample?
2. Build a consensus with peaks supported by at least 3 samples instead of 2 (Step 3b and the end of 3c). How many regions do you get? How many of the induced-only regions do you lose?
3. Compare the two induced replicates with `bdgdiff` (`FOXA1HNF4A_I1` and `FOXA1HNF4A_I2`). How many "differential" regions do you find, and in which direction? Can you predict the direction from the FRiP values of Step 4?